# LingkodAI Stage 7 A/B on Colab: qwen-tts against qwentts.cpp

Branch `tts-cpp`. Synthesizes the same DS6 answers (`rag_answer_native`, in
each row's `final_lang`) with three Stage 7 arms, then scores every arm with
the DS6/DS7 judge:

| Arm | Backend | What it is |
|---|---|---|
| A | `hf` | `src/tts.py`, qwen-tts fp32: the evaluated path |
| B | `cpp`, `parity` | patched qwentts.cpp `tts-server`, same text path as A |
| C | `cpp`, `segmented` | split every language, length cap, retry, text-only on a second failure |
| C on Q8_0 | `cpp`, `segmented` | arm C with Q8_0 weights instead of F32 |

Only Stage 7 changes, so there is no ASR, MT or RAG drift between arms.
`scripts/tts_ab.py` documents the row selection and the scoring; the README's
qwentts.cpp section and `CHANGELOG.md` list every deviation from the
evaluated pipeline.

**Before you start**

- Runtime: GPU, **L4 or A100**. A T4 (sm_75, no bf16, fp16 tensor cores) is
  untested with qwentts.cpp. Arm A is the slow one: about 2.3 times the audio
  length on an A100 (DS6 synthesis notebook), so the default selection
  (about 43 minutes of audio) takes roughly 1.5 to 2.5 hours for arm A alone.
  Lower `PER_STRATUM` in step 7 for a quicker pass.
- A Colab Secret named `HF_TOKEN` with notebook access on: a fine-grained
  read-only token that can read the `lingkodai` ASR model repos (the judge)
  and the `lingkodai/lingkodai-eval-assets` dataset repo.
- Disk: about 11 GB of GGUF weights, the Qwen3-TTS checkpoint, three Whisper
  judges and the eval assets.

The TTS reference clips are Philippine Languages Database speaker audio under
a CC-BY-NC, research-only licence, and every output WAV is a clone of that
speaker. Keep the Drive folder and the WAV zip private.

Run the cells in order. Every result is written straight to your Google
Drive (`MyDrive/lingkodai_tts_cpp_ab/<RUN_NAME>/`, set in step 3), so a
dropped runtime loses no finished rows. To resume: reconnect, run steps 1 to
5 again with the same `RUN_NAME` (step 6 too, unless its report in the run
folder already passed), then step 7 and the step you were on. Finished rows
are skipped.

## 1. Check the GPU

In [ ]:
!nvidia-smi
import torch
print("torch", torch.__version__, "| CUDA available:", torch.cuda.is_available())
assert torch.cuda.is_available(), "No GPU: Runtime > Change runtime type > GPU (L4 or A100)"
print(torch.cuda.get_device_name(0), "| compute capability:", torch.cuda.get_device_capability(0))

## 2. Get the code and install

Same install as the golden-check notebook, plus `utmos-pytorch==0.1.0`, the
UTMOS22 implementation DS6/DS7 was scored with. `qwen-tts` pins
`transformers==4.57.3` and `accelerate==1.12.0`; pip may downgrade Colab's
copies to match, which is expected. Only a `pip check` conflict involving
`transformers`, `accelerate`, `qwen-tts` or `torch` matters here.

In [ ]:
BRANCH = "tts-cpp"
!apt-get -qq update > /dev/null && apt-get -qq install -y ffmpeg sox > /dev/null
import os
!test -d /content/LingkodAI || git clone -q --branch {BRANCH} https://github.com/isabelbeab/LingkodAI.git /content/LingkodAI
assert os.path.isdir("/content/LingkodAI/scripts"), "git clone failed; see the output above"
%cd /content/LingkodAI
!git log --oneline -1
!pip install -q -e ".[gpu,eval]" "utmos-pytorch==0.1.0"
assert _exit_code == 0, "pip install failed; see the output above"
!pip check
!python -c "import torch, transformers, accelerate; print('torch', torch.__version__, '| transformers', transformers.__version__, '| accelerate', accelerate.__version__)"

## 3. Token, environment and Drive

Reads `HF_TOKEN` from Colab Secrets (never printed or written to disk) and
mounts Google Drive now, so the long steps never wait on a permission popup.

`RUN_NAME` names the run folder on Drive where every log, WAV and score
goes. Keep it the same to resume a run; change it to start a new one.

In [ ]:
%cd /content/LingkodAI
import os, re
from google.colab import userdata

os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
assert os.environ["HF_TOKEN"], "The HF_TOKEN secret is empty"
print("HF_TOKEN set")

from google.colab import drive
drive.mount("/content/drive")

RUN_NAME = "run1"
assert re.fullmatch(r"[A-Za-z0-9_.-]+", RUN_NAME), "RUN_NAME: letters, digits, '_', '.', '-' only"
RUN_DIR = f"/content/drive/MyDrive/lingkodai_tts_cpp_ab/{RUN_NAME}"
os.makedirs(RUN_DIR, exist_ok=True)
existing = sorted(os.listdir(RUN_DIR))
print("run folder:", RUN_DIR, "| already holds:", existing or "nothing (new run)")

## 4. Fetch the private eval assets

The TTS reference clips and `DS6_augmented.xlsx`, each checked against the
asset repo's `MANIFEST.sha256`.

In [ ]:
%cd /content/LingkodAI
ASSETS = "/content/LingkodAI/outputs/eval_assets"
!python scripts/fetch_eval_assets.py --out {ASSETS}
assert _exit_code == 0, "fetch_eval_assets.py failed; see the output above"
os.environ["TTS_REF_DIR"] = f"{ASSETS}/tts_ref"
DS6 = f"{ASSETS}/golden/DS6_augmented.xlsx"
assert os.path.isfile(DS6), DS6
print("TTS_REF_DIR =", os.environ["TTS_REF_DIR"], "| DS6 =", DS6)

## 5. Build tts-server and download the weights

`scripts/build_qwentts.sh` clones qwentts.cpp at the pinned commit, applies
`patches/qwentts/` and builds `tts-server`, `qwen-tts` and the C ABI test.
`--cuda-arch native` builds for this runtime's GPU only (a new runtime
rebuilds anyway); the script's default covers T4, A100 and L4. The build
takes several minutes.

`scripts/fetch_qwentts_weights.py` downloads the F32 and Q8_0 GGUFs at a
pinned revision and checks each SHA-256. The Qwen3-TTS checkpoint (arm A, and
the Python side of the patch check) comes from the Hub cache.

In [ ]:
%cd /content/LingkodAI
BUILD_LOG = f"{RUN_DIR}/build_qwentts.log"
!bash scripts/build_qwentts.sh --cuda-arch native > {BUILD_LOG} 2>&1
build_rc = _exit_code
!grep -v '^\[ *[0-9]*%\]' {BUILD_LOG} | tail -40
assert build_rc == 0, f"build failed; see {BUILD_LOG}"
SERVER_BIN = "/content/LingkodAI/build/qwentts.cpp/build/tts-server"
os.environ["QWENTTS_SERVER_BIN"] = SERVER_BIN

GGUF = "/content/models/gguf"
!python scripts/fetch_qwentts_weights.py --out {GGUF}
assert _exit_code == 0, "weight download or checksum failed; see the output above"
WEIGHTS = {q: (f"{GGUF}/qwen-talker-1.7b-base-{q}.gguf", f"{GGUF}/qwen-tokenizer-12hz-{q}.gguf") for q in ("F32", "Q8_0")}

from huggingface_hub import snapshot_download
QWEN3_TTS = snapshot_download("Qwen/Qwen3-TTS-12Hz-1.7B-Base")
print("Qwen3-TTS checkpoint:", QWEN3_TTS)

## 6. Validate the ICL layout patch

`scripts/validate_qwentts_patch.py` runs qwentts.cpp's own cossim harness
(greedy, Python qwen-tts against the C++ binary, same intermediates dumped)
three times: an upstream-layout control, the patched layout on the harness's
English example, and the patched layout with LingkodAI's ceb voice and
`--lang auto`. Pass: prompt ids exact, embedding shapes equal and decomposing
into the expected layout, every text row at cosine 0.9999 or more, prefill
logits at 0.999 or more. Do not run the arms if this fails.

In [ ]:
%cd /content/LingkodAI
REF_DIR = os.environ["TTS_REF_DIR"]
!bash -o pipefail -c "python scripts/validate_qwentts_patch.py --gguf-dir {GGUF} --checkpoint {QWEN3_TTS} --ref-dir {REF_DIR} --out {RUN_DIR}/patch_validation 2>&1 | tee {RUN_DIR}/patch_validation.log"
assert _exit_code == 0, f"patch validation failed; see {RUN_DIR}/patch_validation/"

## 7. Choose the rows

The golden-check conversation (`CONV-MT-001`, 24 turns), `PER_STRATUM` rows
per verbalized-length quartile per language, `ceb_01_mt_185` (DS6's one
runaway) and every DS6 row with round-trip WER above 2. Deterministic for a
given seed. Rerun this cell when resuming: it checks that the run folder's
selection is the same one, and stops if not (a changed `PER_STRATUM` needs a
new `RUN_NAME`).

In [ ]:
%cd /content/LingkodAI
import shutil
import pandas as pd
PER_STRATUM = 3
AB = f"{RUN_DIR}/tts_ab"
SEL = f"{AB}/selection.csv"
!python scripts/tts_ab.py select --ds6 {DS6} --out /content/selection_new.csv --per-stratum {PER_STRATUM}
assert _exit_code == 0
os.makedirs(AB, exist_ok=True)
if os.path.exists(SEL):
    same = pd.read_csv(SEL).clip_id.tolist() == pd.read_csv("/content/selection_new.csv").clip_id.tolist()
    assert same, f"{SEL} holds a different selection; set a new RUN_NAME in step 3 to change it"
    print("Resuming with the selection already in", SEL)
else:
    shutil.copy("/content/selection_new.csv", SEL)
    print("Selection saved to", SEL)

## 8. Arm A: qwen-tts (the evaluated path)

The long step. If the runtime drops, resume as the top of the notebook says:
finished rows are skipped, and a row whose WAV had not reached Drive yet is
synthesized again.

In [ ]:
%cd /content/LingkodAI
!bash -o pipefail -c "python scripts/tts_ab.py synth --ds6 {DS6} --selection {SEL} --arm A --label A-hf --out {AB}/A-hf 2>&1 | tee -a {AB}/A-hf.log"
assert _exit_code == 0

## 9. Arms B and C on F32, and C on Q8_0

Same base seed for every cpp arm, recorded in each run's `run.json`. Each run
also keeps its `tts-server.log`.

In [ ]:
%cd /content/LingkodAI
SEED = 1234
for arm, label, quant in (("B", "B-parity-F32", "F32"), ("C", "C-seg-F32", "F32"), ("C", "C-seg-Q8_0", "Q8_0")):
    talker, codec = WEIGHTS[quant]
    cmd = (f"python scripts/tts_ab.py synth --ds6 {DS6} --selection {SEL} --arm {arm} --label {label}"
           f" --out {AB}/{label} --seed {SEED} --talker-gguf {talker} --codec-gguf {codec}")
    print(cmd)
    !bash -o pipefail -c "{cmd} 2>&1 | tee -a {AB}/{label}.log"
    assert _exit_code == 0, f"{label} failed; see {AB}/{label}.log"

## 10. Score every arm

Judge WER (rows of at most 75 reference words, against DS6's stored
`reference_norm`) and UTMOS22 on every row, plus length, speed and play-ahead
figures. `DS6 (stored)` is the evaluated run's own numbers for the same rows.

In [ ]:
%cd /content/LingkodAI
RUNS = " ".join(f"{AB}/{label}" for label in ("A-hf", "B-parity-F32", "C-seg-F32", "C-seg-Q8_0"))
!bash -o pipefail -c "python scripts/tts_ab.py score --ds6 {DS6} --runs {RUNS} --out {AB}/scores 2>&1 | tee {AB}/score.log"
assert _exit_code == 0

## 11. Read the results

`text_only` is how many turns fell back to text. `ratio_*` is audio length
over the length expected from the verbalized text (DS6's runaway was 3.08).
`ttfa_*` is the TTS-side time to first audio: the whole synthesis for arm A,
the first segment for B and C (play-ahead, simulated). `rows_with_stalls`
counts turns whose simulated playback would pause waiting for a segment.

In [ ]:
%cd /content/LingkodAI
import pandas as pd
pd.set_option("display.float_format", "{:.3f}".format)
display(pd.read_csv(f"{AB}/scores/summary.csv", index_col=0))
display(pd.read_csv(f"{AB}/scores/summary_by_lang.csv", index_col=[0, 1]))
scores = pd.read_csv(f"{AB}/scores/scores.csv")
print("ceb_01_mt_185, DS6's runaway:")
display(scores[scores.clip_id == "ceb_01_mt_185"][["label", "ok", "audio_s", "length_ratio", "utmos", "rtf", "failure_reason"]])
print("Text-only turns:")
display(scores[~scores.ok][["label", "clip_id", "lang", "failure_reason"]])

## 12. Zip the WAVs

Everything is already in the run folder on Drive. This adds
`tts_ab_wavs.zip` there, every arm's WAVs in one file for downloading and
listening. Keep the run folder and the zip private.

In [ ]:
%cd /content/LingkodAI
import zipfile
from pathlib import Path

with zipfile.ZipFile(f"{RUN_DIR}/tts_ab_wavs.zip", "w") as zf:
    for wav in sorted(Path(AB).glob("*/*/*.wav")):
        zf.write(wav, wav.relative_to(AB))
print("wrote", f"{RUN_DIR}/tts_ab_wavs.zip")
!ls -la {RUN_DIR}

## 13. Listen

TTS is judged by listening. One row per golden turn, the arms side by side.
A missing player means that arm fell back to text for the turn (reason in
step 11).

In [ ]:
%cd /content/LingkodAI
from pathlib import Path
from IPython.display import Audio, Markdown, display

sel = pd.read_csv(SEL)
for cid in sel[sel.tags.str.contains("golden|runaway")].clip_id:
    display(Markdown(f"### {cid}"))
    for label in ("A-hf", "B-parity-F32", "C-seg-F32", "C-seg-Q8_0"):
        wav = Path(AB) / label / "wavs" / f"{cid}.wav"
        display(Markdown(f"**{label}**" + ("" if wav.exists() else ": text only")))
        if wav.exists():
            display(Audio(str(wav)))

## 14. Finish

Drive uploads in the background. Run this before closing the runtime: it
waits until every file has reached Drive, then unmounts it (rerun step 3 to
read the run folder again in this runtime).

In [ ]:
from google.colab import drive
drive.flush_and_unmount()
print("All files are on Drive:", RUN_DIR)